# Day 57 — Project: an AutoGen review team

Recreate the capstone's idea as an AutoGen team: a **builder** proposes an image config and
a **security reviewer** must reply `APPROVE` or demand a fix. This is exactly the two-key
review wired into [the capstone](../../phase-9-capstone/avd_imager/reviewer.py).

> Install: `pip install "autogen-agentchat" "autogen-ext[openai]"` and set `OPENAI_API_KEY`. For a local model, pass `base_url=` + `model_info=` to the client.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

mc = OpenAIChatCompletionClient(model="gpt-4o-mini")
builder = AssistantAgent("builder", model_client=mc,
    system_message="Propose a Windows image config (apps + whether Windows Updates are applied).")
reviewer = AssistantAgent("security_reviewer", model_client=mc,
    system_message=("Approve ONLY if Windows Updates are applied and no outdated apps. "
                    "Reply 'APPROVE' if safe, else name the exact risk to fix."))

# TODO: team stops on 'APPROVE' or after 6 messages; run on
#       "Build an image with Teams + an app called HancePro 3.1.0, no Windows Updates."
#       Print the transcript so you can see the reviewer block it.


## 🔒 Solution

Verified against autogen-agentchat 0.7.5.

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

mc = OpenAIChatCompletionClient(model="gpt-4o-mini")
builder = AssistantAgent("builder", model_client=mc,
    system_message="Propose a Windows image config (apps + whether Windows Updates are applied).")
reviewer = AssistantAgent("security_reviewer", model_client=mc,
    system_message=("Approve ONLY if Windows Updates are applied and no outdated apps. "
                    "Reply 'APPROVE' if safe, else name the exact risk to fix."))

team = RoundRobinGroupChat(
    [builder, reviewer],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(6),
)
result = await team.run(
    task="Build an image with Teams + an app called HancePro 3.1.0, with no Windows Updates.")
for m in result.messages:
    print(f"{m.source:18}: {m.content}")
await mc.close()